General analysis of (VisualWebArena) tasks and creation of task subsets based ona root agent.

NOTE: relatively old. Some paths may need adjustment.


In [13]:
import json
import pandas as pd
import numpy as np

def requires_fuzzy_match(test_config):
    if 'eval' not in test_config:
        return False

    if 'reference_answers' in test_config['eval']:
        approach = test_config["eval"]["reference_answers"]
        if approach is not None and "fuzzy_match" in approach:
            return True

    if 'program_html' in test_config['eval']:
        targets = test_config["eval"]["program_html"]
        for target in targets:
            if "fuzzy_match" in target["required_contents"]:
                return True
    return False


def requires_multi_site(test_config):
     if len(test_config['sites']) > 1:
         return True
     return False


def requires_img_input(test_config):
     if "image" in test_config and test_config["image"] is not None:
         return True
     return False

# Function to create a flat list from JSON files
def create_flat_list(input_file):
    with open(input_file, 'r') as file:
        data = json.load(file)
    
    dataset = input_file.split('/')[2].split('-')[0]
    category = input_file.split('/')[3].split('_')[1].split('.')[0]
    
    flat_list = []
    for item in data:
        flat_item = {
            'task_id': item['task_id'],
            'dataset': dataset,
            'category': category,
            'intent_template_id': item['intent_template_id'],
            'reasoning_difficulty': item['reasoning_difficulty'],
            'requires_fuzzy_match': requires_fuzzy_match(item),
            'requires_multi_site': requires_multi_site(item),
            'requires_img_input': requires_img_input(item)
        }
        flat_list.append(flat_item)
    
    return flat_list


def calculate_metrics(df, task_subsets=None, model_start_col=19):
    if task_subsets is not None:
        # Create a boolean mask for filtering
        mask = pd.Series(False, index=df.index)
        for category, ids in task_subsets.items():
            mask |= (df['category'] == category) & (df['task_id'].isin(ids))
        df = df[mask]
    
    # Create a dictionary for aggregation
    agg_dict = {}
    for col in df.columns[model_start_col:]:
        agg_dict[col] = [
            ('mean', lambda x: x.mean(skipna=True)),
            ('count', lambda x: x.count())
        ]
    
    metrics = df.groupby('category').agg(agg_dict).reset_index()
    
    # Flatten column names
    metrics.columns = ['category'] + [f'{col}_{agg}' for col in df.columns[model_start_col:] for agg in ['mean', 'count']]
    return metrics

def load_task_subsets(task_subsets_dir):
    subsets = {}
    for task_category in ['shopping', 'classifieds', 'reddit']:
        with open(f'{task_subsets_dir}/{task_category}.txt', 'r') as file:
            subsets[task_category] = [int(line.strip()) for line in file.readlines()]
    return subsets

def scores_comparison(task_subsets={}, task_subsets_dir= None, include_models=['gpt4o-tree'], scores_per_model_path = '../experiments/scores_per_model.csv', model_start_col=19):  
    if len(task_subsets) == 0:
        task_subsets = load_task_subsets(task_subsets_dir)
    
    scores_per_model_df = pd.read_csv(scores_per_model_path)

    # Calculate metrics for all tasks
    all_tasks_metrics = calculate_metrics(df=scores_per_model_df, task_subsets=None, model_start_col=model_start_col)
    # print(f"Metrics for all tasks:\n{all_tasks_metrics}\n")

    # Calculate metrics for subset tasks
    subset_metrics = calculate_metrics(df=scores_per_model_df, task_subsets=task_subsets, model_start_col=model_start_col)
    # print(f"Metrics for subset tasks:\n{subset_metrics}\n")

    # Compare the differences
    print("==========================================\nDifferences in metrics between all tasks and subset tasks:\n==========================================")
    for category in all_tasks_metrics['category']:
        print(category)
        all_tasks_row = all_tasks_metrics[all_tasks_metrics['category'] == category].iloc[0]
        subset_row = subset_metrics[subset_metrics['category'] == category].iloc[0]
        
        for col in all_tasks_metrics.columns[1:]:
            if col.endswith('_mean'):
                model = '_'.join(col.split('_')[:-1])  # Join all parts except the last one
                
                if len(include_models) > 0 and not any(include_model in model for include_model in include_models):
                    continue
                if f'{model}_mean' in all_tasks_row.index and f'{model}_mean' in subset_row.index:
                    all_mean = all_tasks_row[f'{model}_mean']
                    subset_mean = subset_row[f'{model}_mean']
                    all_count = all_tasks_row[f'{model}_count']
                    subset_count = subset_row[f'{model}_count']
                    
                    print(f"  {model}:")
                    print(f"    Difference in mean score: {subset_mean - all_mean:.4f}")
                    print(f"    All tasks: mean = {all_mean:.4f}, count = {all_count}")
                    print(f"    Subset:    mean = {subset_mean:.4f}, count = {subset_count}")
                else:
                    print(f"  {model}: Data not available for comparison")

def calculate_distribution_per_attribute(df, attribute='reasoning_difficulty', weight_column=None):
    if weight_column:
        # Weighted distribution based on 'n_samples'
        difficulty_counts = df.groupby(attribute)[weight_column].sum()
    else:
        # Simple distribution count
        difficulty_counts = df[attribute].value_counts()

    # Calculate the total number of tasks (or weighted sum)
    total_tasks = difficulty_counts.sum()

    # Calculate the distribution as proportions
    difficulty_distribution = (difficulty_counts / total_tasks).to_dict()

    return difficulty_distribution


In [4]:
# Function to REDUCE the number of samples in the dataset until the desired number of samples is reached
# Tries to keep the distribution of reasoning difficulties close to the original distribution
def reduce_sample_size(samples_per_attributes, original_distribution, desired_total_samples, keep_at_least_one_intent):
    """Adjust the sample size to match the original distribution."""
    current_total = samples_per_attributes['n_samples'].sum()

    while current_total > desired_total_samples:
        retries=0
        # Calculate current distribution of sampled data
        current_distribution = calculate_distribution_per_attribute(samples_per_attributes.loc[samples_per_attributes['n_samples'] > 0], 'reasoning_difficulty', 'n_samples')
        
        # Compute the difference between the current distribution and the original distribution; sort by discrepancy
        diff = pd.Series(current_distribution) - pd.Series(original_distribution)
        diff = diff.sort_values(ascending=False)

        # Identify the groups in the category to reduce
        category_to_reduce = diff.index[0]
        groups_to_reduce = samples_per_attributes[samples_per_attributes['reasoning_difficulty'] == category_to_reduce]

        # Sort by the number of samples to reduce from the largest group first
        max_group_index = groups_to_reduce['n_samples'].idxmax()

        # If the group has more than one sample, reduce the number of samples
        if samples_per_attributes.loc[max_group_index, 'n_samples'] > 1:
            samples_per_attributes.loc[max_group_index, 'n_samples'] -= 1
            current_total =  samples_per_attributes['n_samples'].sum()
            continue

        # Otherwise, remove the group or reduce the next group
        if not keep_at_least_one_intent:
            # If we don't need to keep at least one intent, we can remove the group
            samples_per_attributes = samples_per_attributes.drop(max_group_index)
            current_total = samples_per_attributes['n_samples'].sum()
        # Try to reduce the following groups in order of discrepancy to original distribution
        else: 
            retries += 1
            while retries < len(groups_to_reduce)-1:
                if diff[retries] < 0:
                    retries = len(groups_to_reduce)-1
                    break
    
                category_to_reduce = diff.index[retries]
                groups_to_reduce = samples_per_attributes[samples_per_attributes['reasoning_difficulty'] == category_to_reduce]

                max_group_index = groups_to_reduce['n_samples'].idxmax()
                if samples_per_attributes.loc[max_group_index, 'n_samples'] > 1:
                    samples_per_attributes.loc[max_group_index, 'n_samples'] -= 1
                    current_total = samples_per_attributes['n_samples'].sum()
                    break
                retries += 1

        if retries == len(groups_to_reduce)-1:
            break
    return samples_per_attributes

In [5]:
raw_test_files = [
    '../config_files/vwa/test_shopping.json',
    '../config_files/vwa/test_classifieds.json',
    '../config_files/vwa/test_reddit.json',
]

all_tasks = []
for test_file in raw_test_files:
    all_tasks.extend(create_flat_list(test_file))

all_tasks_df = pd.DataFrame(all_tasks)
proportion_per_task_category = all_tasks_df['category'].value_counts(normalize=True).to_dict()

proportion_per_task_category

{'shopping': 0.512087912087912,
 'classifieds': 0.2571428571428571,
 'reddit': 0.23076923076923078}

In [6]:
overall_num_samples = len(all_tasks_df) // 3
print(f'Overall number of samples: {overall_num_samples}')

num_samples_per_task_category = {k: int(v * overall_num_samples) for k, v in proportion_per_task_category.items()}

print(f"Necessary samples per task category: {num_samples_per_task_category}")


desired_samples_per_task_category = {
    'shopping': 100,
    'classifieds': 80,
    'reddit': 70
}

print(f'Desired samples per task category: {desired_samples_per_task_category}')

keep_at_least_one_intent = False

exclude_fuzzy_match = False
exclude_multi_site = False
exclude_img_input = False

Overall number of samples: 303
Necessary samples per task category: {'shopping': 155, 'classifieds': 77, 'reddit': 69}
Desired samples per task category: {'shopping': 100, 'classifieds': 80, 'reddit': 70}


In [7]:
random_seed = 20

subsets = {}
for test_file in raw_test_files:
    print(f"============================\n\nSubsampling tasks from: {test_file}\n\n============================")
    df = pd.DataFrame(create_flat_list(test_file))

    if exclude_fuzzy_match:
        df = df[df['requires_fuzzy_match'] == False]
    if exclude_multi_site:
        df = df[df['requires_multi_site'] == False]
    if exclude_img_input:
        df = df[df['requires_img_input'] == False]

    # Get task category ('shopping', 'classifieds', 'reddit', etc)
    task_category = df['category'].unique()[0]

    # Get desired number of samples for the task category
    desired_total_samples=desired_samples_per_task_category[task_category]

    # Calculate original reasoning difficulty distribution
    original_distribution = calculate_distribution_per_attribute(df, 'reasoning_difficulty')

    # Calculate the number of samples per [reasoning_difficulty, intent_template_id] attributes required
    samples_per_attributes = df.groupby(['reasoning_difficulty', 'intent_template_id']).size().reset_index(name='counts')
    samples_per_attributes['proportion'] = samples_per_attributes['counts'] / samples_per_attributes['counts'].sum()

    # Number of sample for each [reasoning_difficulty, intent_template_id] for the new dataset, with a minimum of 1 sample
    samples_per_attributes['n_samples'] = np.round(np.maximum(samples_per_attributes['proportion'] * desired_total_samples, 1)).astype(int)

    # Total samples if requires at least one sample per [reasoning_difficulty, intent_template_id]
    temp_size = samples_per_attributes['n_samples'].sum()

    # Reduce the number of samples in the dataset until the desired number of samples is reached
    if len(samples_per_attributes) > desired_total_samples:
        samples_per_attributes = reduce_sample_size(samples_per_attributes, original_distribution, desired_total_samples, keep_at_least_one_intent)

    print("Original size:", len(df))
    print("Desired size:", desired_total_samples)
    print("Size first round", temp_size)
    print("Final Subset size:", samples_per_attributes['n_samples'].sum())

    # Check the distribution before sampling
    proportion_difficulty =  pd.Series(calculate_distribution_per_attribute(df, 'reasoning_difficulty')).sort_index().to_dict()
    print("Original distribution:\n", proportion_difficulty)

    # Check the distribution after subsampling
    proportion_subsample = pd.Series(calculate_distribution_per_attribute(samples_per_attributes.loc[samples_per_attributes['n_samples'] > 0], 'reasoning_difficulty', 'n_samples')).sort_index().to_dict()
    print("Adjusted distribution:\n", proportion_subsample)

    # Perform the actual sampling based on the adjusted samples_per_attributes
    sampled_df = pd.DataFrame()  # Initialize an empty DataFrame to store our samples


    for _, row in samples_per_attributes.iterrows():
        # Filter the original dataframe based on reasoning_difficulty and intent_template_id
        group_df = df[(df['reasoning_difficulty'] == row['reasoning_difficulty']) & 
                    (df['intent_template_id'] == row['intent_template_id'])]

        # Sample n_samples from this group
        if row['n_samples'] > 0:
            group_sample = group_df.sample(n=min(row['n_samples'], len(group_df)), random_state=random_seed)
            sampled_df = pd.concat([sampled_df, group_sample], ignore_index=True)

    # Now sampled_df contains our actual samples
    print(f"Total samples: {len(sampled_df)}")

    # Store the task IDs
    subsets[task_category] = sorted(sampled_df['task_id'].tolist())


Subsampling tasks from: ../config_files/vwa/test_shopping.json

Original size: 466
Desired size: 100
Size first round 200
Final Subset size: 100
Original distribution:
 {'easy': 0.36909871244635195, 'hard': 0.2832618025751073, 'medium': 0.34763948497854075}
Adjusted distribution:
 {'easy': 0.37, 'hard': 0.28, 'medium': 0.35}
Total samples: 100

Subsampling tasks from: ../config_files/vwa/test_classifieds.json

Original size: 234
Desired size: 80
Size first round 122
Final Subset size: 80
Original distribution:
 {'easy': 0.2905982905982906, 'hard': 0.38461538461538464, 'medium': 0.3247863247863248}
Adjusted distribution:
 {'easy': 0.2875, 'hard': 0.3875, 'medium': 0.325}
Total samples: 80

Subsampling tasks from: ../config_files/vwa/test_reddit.json

Original size: 210
Desired size: 70
Size first round 107
Final Subset size: 70
Original distribution:
 {'easy': 0.2761904761904762, 'hard': 0.3476190476190476, 'medium': 0.3761904761904762}
Adjusted distribution:
 {'easy': 0.27142857142857

In [16]:
task_subsets = subsets
task_subsets_dir = None # '../evaluation_harness/task_subsets'

scores_comparison(task_subsets=task_subsets, task_subsets_dir=task_subsets_dir,
                   include_models=['gpt4o-tree','gemini-1.5-002-flash_base-stateaware'],
                   model_start_col=19)


Differences in metrics between all tasks and subset tasks:
classifieds
  gpt4o-tree:
    Difference in mean score: 0.0127
    All tasks: mean = 0.2972, count = 212
    Subset:    mean = 0.3099, count = 71
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: 0.0002
    All tasks: mean = 0.1361, count = 191
    Subset:    mean = 0.1364, count = 66
reddit
  gpt4o-tree:
    Difference in mean score: 0.0309
    All tasks: mean = 0.2079, count = 202
    Subset:    mean = 0.2388, count = 67
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: 0.0167
    All tasks: mean = 0.1371, count = 124
    Subset:    mean = 0.1538, count = 39
shopping
  gpt4o-tree:
    Difference in mean score: 0.0161
    All tasks: mean = 0.2932, count = 457
    Subset:    mean = 0.3093, count = 97
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: 0.0162
    All tasks: mean = 0.1738, count = 466
    Subset:    mean = 0.1900, count = 100


In [23]:
# Write task ids to text files
import os

dir_to_save = '../evaluation_harness/task_subsets_smaller'

os.makedirs(dir_to_save, exist_ok=True)
for task_category, task_ids in subsets.items():
    with open(f'{dir_to_save}/{task_category}.txt', 'w') as file:
        for task_id in task_ids:
            file.write(f'{task_id}\n')

Analysis if the distribution of reasoning difficulties is similar to the original distribution

In [15]:
import pandas as pd

def calculate_metrics(df, task_subsets=None, model_start_col=20):
    if task_subsets is not None:
        # Create a boolean mask for filtering
        mask = pd.Series(False, index=df.index)
        for category, ids in task_subsets.items():
            mask |= (df['category'] == category) & (df['task_id'].isin(ids))
        df = df[mask]
    
    # Create a dictionary for aggregation
    agg_dict = {}
    for col in df.columns[model_start_col:]:
        agg_dict[col] = [
            ('mean', lambda x: x.mean(skipna=True)),
            ('count', lambda x: x.count())
        ]
    
    metrics = df.groupby('category').agg(agg_dict).reset_index()
    
    # Flatten column names
    metrics.columns = ['category'] + [f'{col}_{agg}' for col in df.columns[model_start_col:] for agg in ['mean', 'count']]
    return metrics

def load_task_subsets(task_subsets_dir):
    subsets = {}
    for task_category in ['shopping', 'classifieds', 'reddit']:
        with open(f'{task_subsets_dir}/{task_category}.txt', 'r') as file:
            subsets[task_category] = [int(line.strip()) for line in file.readlines()]
    return subsets
include_models =  ['gpt4o-tree','gemini-1.5-002-flash_base-stateaware']

def scores_comparison(task_subsets={}, task_subsets_dir= None, include_models=['gpt4o-tree'], scores_per_model_path = '../experiments/scores_per_model.csv', start_col=19):  
    if len(task_subsets) == 0:
        task_subsets = load_task_subsets(task_subsets_dir)
    
    scores_per_model_df = pd.read_csv(scores_per_model_path)

    # Calculate metrics for all tasks
    all_tasks_metrics = calculate_metrics(scores_per_model_df)
    # print(f"Metrics for all tasks:\n{all_tasks_metrics}\n")

    # Calculate metrics for subset tasks
    subset_metrics = calculate_metrics(scores_per_model_df, task_subsets)
    # print(f"Metrics for subset tasks:\n{subset_metrics}\n")

    # Compare the differences
    print("==========================================\nDifferences in metrics between all tasks and subset tasks:\n==========================================")
    for category in all_tasks_metrics['category']:
        print(category)
        all_tasks_row = all_tasks_metrics[all_tasks_metrics['category'] == category].iloc[0]
        subset_row = subset_metrics[subset_metrics['category'] == category].iloc[0]
        
        for col in all_tasks_metrics.columns[1:]:
            if col.endswith('_mean'):
                model = '_'.join(col.split('_')[:-1])  # Join all parts except the last one
                
                if len(include_models) > 0 and not any(include_model in model for include_model in include_models):
                    continue
                if f'{model}_mean' in all_tasks_row.index and f'{model}_mean' in subset_row.index:
                    all_mean = all_tasks_row[f'{model}_mean']
                    subset_mean = subset_row[f'{model}_mean']
                    all_count = all_tasks_row[f'{model}_count']
                    subset_count = subset_row[f'{model}_count']
                    
                    print(f"  {model}:")
                    print(f"    Difference in mean score: {subset_mean - all_mean:.4f}")
                    print(f"    All tasks: mean = {all_mean:.4f}, count = {all_count}")
                    print(f"    Subset:    mean = {subset_mean:.4f}, count = {subset_count}")
                else:
                    print(f"  {model}: Data not available for comparison")


Differences in metrics between all tasks and subset tasks:
classifieds
  gpt4o-tree:
    Difference in mean score: 0.0223
    All tasks: mean = 0.2972, count = 212
    Subset:    mean = 0.3194, count = 72
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: 0.0608
    All tasks: mean = 0.1361, count = 191
    Subset:    mean = 0.1970, count = 66
reddit
  gpt4o-tree:
    Difference in mean score: 0.0095
    All tasks: mean = 0.2079, count = 202
    Subset:    mean = 0.2174, count = 69
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: -0.0055
    All tasks: mean = 0.1371, count = 124
    Subset:    mean = 0.1316, count = 38
shopping
  gpt4o-tree:
    Difference in mean score: -0.0046
    All tasks: mean = 0.2932, count = 457
    Subset:    mean = 0.2887, count = 97
  gemini-1.5-002-flash_base-stateaware:
    Difference in mean score: -0.0342
    All tasks: mean = 0.1787, count = 431
    Subset:    mean = 0.1444, count = 90


In [15]:
# Load the task subset IDs
task_subsets = {}
task_subsets_dir = '../evaluation_harness/task_subsets'

for task_category in ['shopping', 'classifieds', 'reddit']:
    with open(f'{task_subsets_dir}/{task_category}.txt', 'r') as file:
       task_subsets[task_category] = [int(line.strip()) for line in file.readlines()]

# Calculate total number of tasks in the original dataset
total_original_tasks = sum(all_tasks_df['category'].value_counts())

print(f"Total number of tasks in the original dataset: {total_original_tasks}")
print("\nOriginal proportion of task categories:")
for category, proportion in proportion_per_task_category.items():
    num_tasks = all_tasks_df['category'].value_counts()[category]
    print(f"  {category}: {proportion:.4f} ({num_tasks} tasks)")

# Calculate the total number of tasks in subsets
total_subset_tasks = sum(len(tasks) for tasks in task_subsets.values())

print(f"\nTotal number of tasks in the subsets: {total_subset_tasks}")
print("\nProportion in the subsets:")
# Calculate and print the proportion for each category in the subsets
for category, tasks in task_subsets.items():
    subset_proportion = len(tasks) / total_subset_tasks
    print(f"  {category}: {subset_proportion:.4f} ({len(tasks)} tasks)")


Total number of tasks in the original dataset: 910

Original proportion of task categories:
  shopping: 0.5121 (466 tasks)
  classifieds: 0.2571 (234 tasks)
  reddit: 0.2308 (210 tasks)

Total number of tasks in the subsets: 305

Proportion in the subsets:
  shopping: 0.5115 (156 tasks)
  classifieds: 0.2590 (79 tasks)
  reddit: 0.2295 (70 tasks)


In [17]:
import pandas as pd

# Load the task subset IDs
task_subsets = {}
task_subsets_dir = '../evaluation_harness/task_subsets'

for task_category in ['shopping', 'classifieds', 'reddit']:
    with open(f'{task_subsets_dir}/{task_category}.txt', 'r') as file:
       task_subsets[task_category] = [int(line.strip()) for line in file.readlines()]

def get_category_difficulty_breakdown(df):
    breakdown = df.groupby(['category', 'reasoning_difficulty']).size().unstack(fill_value=0)
    breakdown_percentages = breakdown.div(breakdown.sum(axis=1), axis=0) * 100
    breakdown_percentages = breakdown_percentages.round(2)
    
    # Combine counts and percentages
    combined = pd.DataFrame()
    for col in breakdown.columns:
        combined[f'{col} Count'] = breakdown[col]
        combined[f'{col} %'] = breakdown_percentages[col]
    
    combined = combined.sort_index()
    combined['Total'] = breakdown.sum(axis=1)
    return combined

# Original dataset breakdown
print("Original Dataset Breakdown:")
original_breakdown = get_category_difficulty_breakdown(all_tasks_df)
print(original_breakdown.to_string())
print(f"\nTotal tasks in original dataset: {len(all_tasks_df)}")

# Subset breakdown
subset_df = all_tasks_df[all_tasks_df['task_id'].isin([id for ids in task_subsets.values() for id in ids])]
print("\nSubset Breakdown:")
subset_breakdown = get_category_difficulty_breakdown(subset_df)
print(subset_breakdown.to_string())
print(f"\nTotal tasks in subset: {len(subset_df)}")

# Overall reasoning difficulty distribution
print("\nOverall Reasoning Difficulty Distribution:")
original_difficulty_dist = all_tasks_df['reasoning_difficulty'].value_counts(normalize=True).sort_index() * 100
subset_difficulty_dist = subset_df['reasoning_difficulty'].value_counts(normalize=True).sort_index() * 100

difficulty_comparison = pd.DataFrame({
    'Original %': original_difficulty_dist,
    'Subset %': subset_difficulty_dist
}).round(2)

print(difficulty_comparison.to_string())

Original Dataset Breakdown:
             easy Count  easy %  hard Count  hard %  medium Count  medium %  Total
category                                                                          
classifieds          68   29.06          90   38.46            76     32.48    234
reddit               58   27.62          73   34.76            79     37.62    210
shopping            172   36.91         132   28.33           162     34.76    466

Total tasks in original dataset: 910

Subset Breakdown:
             easy Count  easy %  hard Count  hard %  medium Count  medium %  Total
category                                                                          
classifieds          43   29.45          54   36.99            49     33.56    146
reddit               44   33.85          42   32.31            44     33.85    130
shopping             90   36.00          77   30.80            83     33.20    250

Total tasks in subset: 526

Overall Reasoning Difficulty Distribution:
             